### ETL para gerar .parquet e subir no git 
#### na pratica se for replicar por ai, so salvar o .parquet e ser feliz, caso queira fazer por si proprio, eis o ETL de exemplo: 

In [6]:
import pandas as pd
import glob
import os

# --- pasta com os CSVs ---
pasta_csv = r"C:\Users\jgeov\Downloads\Help_desk_GLPI_IFPR"
arquivos_csv = glob.glob(os.path.join(pasta_csv, "*.csv"))

# --- concatena todos os CSVs ---
lista_df = []
for arquivo in arquivos_csv:
    df = pd.read_csv(arquivo, encoding="utf-8", sep=";", engine="python")  # engine python tolera quebras de linha
    lista_df.append(df)

df_total = pd.concat(lista_df, ignore_index=True)

# --- Seleciona ---
colunas_importantes = [
    'Ticker', 'Requerente', 'Grupo', 'Categoria',
    'Data Abertura', 'Data Solução', 'Data Fechamento',
    'Atendente', 'Satisfação', 'Data Iteração',
    'Usuário Iteração', 'Resolvido'
]
df_total = df_total[colunas_importantes]

# ---  Cria coluna de texto consolidado para NLP ---

df_total["texto"] = (
    df_total["Categoria"].fillna("").astype(str) + " " +
    df_total["Grupo"].fillna("").astype(str) + " " +
    df_total["Requerente"].fillna("").astype(str) + " " +
    df_total["Atendente"].fillna("").astype(str) + " " +
    df_total["Ticker"].fillna("").astype(str)
)

# --- Filtra apenas linhas com texto e categoria (Grupo) 
df_total = df_total[(df_total["texto"].str.strip() != "") & (df_total["Grupo"].notna())]

# --- Remove duplicatas (com base em texto e Grupo) ---
df_total = df_total.drop_duplicates(subset=["texto", "Grupo"])

#--- trata e trasnforma datas ---

colunas_datas = ["Data Abertura", "Data Solução", "Data Fechamento", "Data Iteração"]

# 1. Converter para datetime
for col in colunas_datas:
    df_total[col] = pd.to_datetime(df_total[col], format="%d/%m/%Y %H:%M", errors="coerce")

# 2. Função para extrair features
def extrair_features(df, coluna):
    prefixo = coluna.replace(" ", "_")  # exemplo: "Data Abertura" -> "Data_Abertura"
    
    df[f"{prefixo}_Ano"] = df[coluna].dt.year
    df[f"{prefixo}_Mes"] = df[coluna].dt.month
    df[f"{prefixo}_Dia"] = df[coluna].dt.day
    df[f"{prefixo}_DiaSemana"] = df[coluna].dt.day_name(locale="pt_BR.utf8")
    df[f"{prefixo}_Hora"] = df[coluna].dt.hour
    #df[f"{prefixo}_Minuto"] = df[coluna].dt.minute
    df[f"{prefixo}_FimDeSemana"] = df[coluna].dt.dayofweek.isin([5,6]).astype(int)

    # Período do dia
    def periodo_do_dia(hora):
        if pd.isna(hora): 
            return None
        if 0 <= hora < 6:
            return "Madrugada"
        elif 6 <= hora < 12:
            return "Manhã"
        elif 12 <= hora < 18:
            return "Tarde"
        else:
            return "Noite"

    df[f"{prefixo}_Periodo"] = df[f"{prefixo}_Hora"].apply(periodo_do_dia)

    return df

# 3. Aplicar para cada coluna de data
for col in colunas_datas:
    df_total = extrair_features(df_total, col)


df_total.head(5)


,Ticker,Requerente,Grupo,Categoria,Data Abertura,Data Solução,Data Fechamento,Atendente,Satisfação,Data Iteração,...,Data_Fechamento_Hora,Data_Fechamento_FimDeSemana,Data_Fechamento_Periodo,Data_Iteração_Ano,Data_Iteração_Mes,Data_Iteração_Dia,Data_Iteração_DiaSemana,Data_Iteração_Hora,Data_Iteração_FimDeSemana,Data_Iteração_Periodo
0,16749,ELIDIONETE DE ANDRADE,SIG_TECNICO,-,2017-12-29 16:43:00,2018-01-09 11:30:00,2018-01-11 10:21:00,TATIANA MAYUMI NIWA,-,2018-01-04 08:42:00,...,10.0,0,Manhã,2018.0,1.0,4.0,Quinta-feira,8.0,0,Manhã
1,16749,ELIDIONETE DE ANDRADE,SIGAA,-,2017-12-29 16:43:00,2018-01-09 11:30:00,2018-01-11 10:21:00,TATIANA MAYUMI NIWA,-,2018-01-04 08:42:00,...,10.0,0,Manhã,2018.0,1.0,4.0,Quinta-feira,8.0,0,Manhã
4,16748,LARISSA DINIZ RIBEIRO,SIGAA,-,2017-12-29 14:22:00,2018-01-02 09:32:00,2018-01-04 08:29:00,TATIANA MAYUMI NIWA,-,NaT,...,8.0,0,Manhã,NaN,NaN,NaN,NaN,NaN,0,None
5,16747,FABIANA APARECIDA PEREIRA DA SILVA,SIG_TECNICO,-,2017-12-29 11:55:00,2018-02-02 13:57:00,2018-02-02 13:57:00,LUCIANA WISTUBA COSMO DE SIQUEIRA E SILVA,-,2017-12-29 13:20:00,...,13.0,0,Tarde,2017.0,12.0,29.0,Sexta-feira,13.0,0,Tarde
6,16747,FABIANA APARECIDA PEREIRA DA SILVA,SIGAA,-,2017-12-29 11:55:00,2018-02-02 13:57:00,2018-02-02 13:57:00,LUCIANA WISTUBA COSMO DE SIQUEIRA E SILVA,-,2017-12-29 13:20:00,...,13.0,0,Tarde,2017.0,12.0,29.0,Sexta-feira,13.0,0,Tarde


In [7]:
colunas_dropar = ['Ticker', 'Requerente', 'Atendente']+colunas_datas
colunas_dropar

['Ticker',
 'Requerente',
 'Atendente',
 'Data Abertura',
 'Data Solução',
 'Data Fechamento',
 'Data Iteração']

In [8]:
## dropa colunas de dadas originais

df_total = df_total.drop(columns=colunas_dropar)

df_total.columns

Index(['Grupo', 'Categoria', 'Satisfação', 'Usuário Iteração', 'Resolvido',
       'texto', 'Data_Abertura_Ano', 'Data_Abertura_Mes', 'Data_Abertura_Dia',
       'Data_Abertura_DiaSemana', 'Data_Abertura_Hora',
       'Data_Abertura_FimDeSemana', 'Data_Abertura_Periodo',
       'Data_Solução_Ano', 'Data_Solução_Mes', 'Data_Solução_Dia',
       'Data_Solução_DiaSemana', 'Data_Solução_Hora',
       'Data_Solução_FimDeSemana', 'Data_Solução_Periodo',
       'Data_Fechamento_Ano', 'Data_Fechamento_Mes', 'Data_Fechamento_Dia',
       'Data_Fechamento_DiaSemana', 'Data_Fechamento_Hora',
       'Data_Fechamento_FimDeSemana', 'Data_Fechamento_Periodo',
       'Data_Iteração_Ano', 'Data_Iteração_Mes', 'Data_Iteração_Dia',
       'Data_Iteração_DiaSemana', 'Data_Iteração_Hora',
       'Data_Iteração_FimDeSemana', 'Data_Iteração_Periodo'],
      dtype='object')

In [9]:
df_total = df_total

In [10]:

# --- Salva em Parquet  ---
saida_parquet = "ifpr_helpdesk_completo.parquet"
df_total.to_parquet(saida_parquet, index=False)

print(f"Base filtrada e salva em Parquet com todas as colunas: {saida_parquet}")

Base filtrada e salva em Parquet com todas as colunas: ifpr_helpdesk_completo.parquet
